# Import libraries

In [1]:
!pip install -q datasets transformers rank-bm25 sentence-transformers tqdm evaluate torch
!pip install numpy --upgrade

# Connect to Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Utility functions

In [3]:
from datasets import load_dataset
import json
import os

base_path = "/content/drive/MyDrive/CapstoneProject/Retriever"

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def load_qrels(path):
    qrels = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            qid, _, docid, rel = line.strip().split()
            qrels.setdefault(qid, {})[docid] = int(rel)
    return qrels

def normalize_scores(scores_dict):
    if not scores_dict: return {}
    vals = list(scores_dict.values())
    min_v, max_v = min(vals), max(vals)
    return {k: (v - min_v) / (max_v - min_v + 1e-9) for k, v in scores_dict.items()}

# Import dataset if not exists

In [4]:
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for d in items:
            f.write(json.dumps(d, ensure_ascii=False) + "\n")

def save_qrels(qrels_list, path):
    # qrels_list là list của (query_id, doc_id, rel)
    with open(path, "w", encoding="utf-8") as f:
        # mỗi dòng: query_id \t 0 \t doc_id \t relevance
        for qid, docid, rel in qrels_list:
            f.write(f"{qid}\t0\t{docid}\t{rel}\n")

def convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val")):
    ds = load_dataset("microsoft/ms_marco", 'v2.1', split=split)
    os.makedirs(out_dir, exist_ok=True)

    # Queries
    queries = [{"id": str(rec["query_id"]), "text": rec["query"]} for rec in ds]

    # Corpus & Qrels
    corpus_map = {}
    qrels = []

    for rec in ds:
        qid = str(rec["query_id"])
        passages = rec["passages"]
        texts = passages["passage_text"]
        selected = passages["is_selected"]

        for idx, text in enumerate(texts):
            if not text:
                continue
            doc_id = f"{qid}_{idx}"
            corpus_map[doc_id] = text
            if selected[idx] == 1:
                qrels.append((qid, doc_id, 1))

    corpus = [{"id": did, "text": txt} for did, txt in corpus_map.items()]

    # Save
    save_jsonl(corpus, os.path.join(out_dir, "corpus.jsonl"))
    save_jsonl(queries, os.path.join(out_dir, "queries.jsonl"))
    save_qrels(qrels, os.path.join(out_dir, "qrels.tsv"))

    print("Done convert MS MARCO split:", split)

# if exists the folder, skip
if not os.path.exists(os.path.join(base_path, "data/ms_marco_val/corpus.jsonl")):
    convert_ms_marco(split="validation", out_dir=os.path.join(base_path, "data/ms_marco_val"))

if not os.path.exists(os.path.join(base_path, "data/ms_marco_train/corpus.jsonl")):
    convert_ms_marco(split="train", out_dir=os.path.join(base_path, "data/ms_marco_train"))


# Implementing retrievers

## BM25 retriever

In [5]:
from rank_bm25 import BM25Okapi

class BM25Retriever:
    def __init__(self, corpus):
        self.doc_ids = [doc["id"] for doc in corpus]
        tokenized_corpus = [doc["text"].split() for doc in tqdm(corpus, desc="Tokenizing corpus for BM25")]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def search(self, query, k=10):
        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)
        top_k_indices = np.argpartition(scores, -k)[-k:]
        top_k_sorted_indices = top_k_indices[np.argsort(scores[top_k_indices])][::-1]
        return [(self.doc_ids[i], float(scores[i])) for i in top_k_sorted_indices]

## Dense retriever

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import os
import torch

class DenseRetriever:
    def __init__(self, corpus, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cuda", embeddings_path=None):
        """
        Khởi tạo DenseRetriever.
        Sẽ ưu tiên tải embeddings từ `embeddings_path` nếu file tồn tại.
        Nếu không, nó sẽ encode corpus và lưu kết quả vào `embeddings_path`.
        """
        self.model = SentenceTransformer(model_name, device=device)
        self.doc_ids = [doc["id"] for doc in corpus]

        if embeddings_path and os.path.exists(embeddings_path):
            print(f"Loading pre-computed embeddings from {embeddings_path}...")
            self.doc_vecs = np.load(embeddings_path)
            print("Embeddings loaded successfully.")
        else:
            print("No pre-computed embeddings found. Encoding corpus (this may take a long time)...")
            texts = [doc["text"] for doc in corpus]
            self.doc_vecs = self.model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=True,
                convert_to_numpy=True
            )
            if embeddings_path:
                print(f"Saving embeddings to {embeddings_path} for future use...")
                np.save(embeddings_path, self.doc_vecs)
                print(f"New embeddings saved.")

        # Chuyển doc_vecs sang GPU để tính toán nhanh hơn
        if torch.cuda.is_available():
            self.doc_vecs_gpu = torch.from_numpy(self.doc_vecs).to(device)

    def search(self, query, k=10):
        q_vec = self.model.encode([query], normalize_embeddings=True)[0]
        sims = np.dot(self.doc_vecs, q_vec)
        top_idx = np.argsort(sims)[-k:][::-1]
        return [(self.doc_ids[i], float(sims[i])) for i in top_idx]

    # Bản này bị tràn RAM
    # def search_batch(self, queries, k=10, batch_size=128):
    #     """Return results for multiple queries efficiently."""
    #     q_vecs = self.model.encode(
    #         queries,
    #         batch_size=batch_size,
    #         normalize_embeddings=True,
    #         show_progress_bar=True,
    #         convert_to_numpy=True
    #     )
    #     sims = np.dot(q_vecs, self.doc_vecs.T)  # this is the culprit
    #     results = []
    #     for row in sims:
    #         top_idx = np.argsort(row)[-k:][::-1]
    #         results.append([(self.doc_ids[i], float(row[i])) for i in top_idx])
    #     return results

    def search_batch(self, queries, k=10, batch_size=64):
        """
        Xử lý batch hiệu quả về bộ nhớ bằng cách chia nhỏ các query.
        """
        results = []
        for i in tqdm(range(0, len(queries), batch_size), desc="Dense Batch Search"):
            queries_chunk = queries[i:i+batch_size]

            # 1. Encode chunk query hiện tại
            q_vecs_chunk = self.model.encode(
                queries_chunk,
                normalize_embeddings=True,
                show_progress_bar=False, # Tắt progress bar con
                convert_to_tensor=True # Chuyển sang Tensor để tính trên GPU
            )

            # 2. Nhân ma trận chunk query với TOÀN BỘ doc vectors trên GPU
            # Phép tính này (ví dụ: 128x384 * 384x1M) sẽ tạo ra ma trận (128, 1M) -> tốn ít RAM hơn nhiều
            sims_chunk = torch.matmul(q_vecs_chunk, self.doc_vecs_gpu.T)

            # Chuyển kết quả về CPU để xử lý
            sims_chunk_cpu = sims_chunk.cpu().numpy()

            # 3. Lấy top-k cho từng query trong chunk
            for row in sims_chunk_cpu:
                top_idx = np.argpartition(row, -k)[-k:]
                top_k_sorted_indices = top_idx[np.argsort(row[top_idx])][::-1]
                results.append([(self.doc_ids[j], float(row[j])) for j in top_k_sorted_indices])

        return results

# Evaluation

In [42]:
corpus = load_jsonl(os.path.join(base_path, "data/ms_marco_val/corpus.jsonl"))
queries = load_jsonl(os.path.join(base_path, "data/ms_marco_val/queries.jsonl"))
qrels = load_qrels(os.path.join(base_path, "data/ms_marco_val/qrels.tsv"))

# take only 5000 queries
queries = queries[:1000]

bm25 = BM25Retriever(corpus)


Tokenizing corpus for BM25: 100%|██████████| 1008985/1008985 [00:09<00:00, 104160.99it/s]


In [8]:
import math
import json
import numpy as np
from tqdm import tqdm

def recall_at_k(run, qrels, k):
    # run: list of (docid, score)
    relevant = {d for d, rel in qrels.items() if rel > 0}
    retrieved = {d for d, _ in run[:k]}
    if not relevant:
        return 0.0
    return len(relevant & retrieved) / len(relevant)

def dcg_at_k(run, qrels, k):
    import math
    return sum(
        (2**qrels.get(doc, 0) - 1) / math.log2(idx + 2)
        for idx, (doc, _) in enumerate(run[:k])
    )

def ndcg_at_k(run, qrels, k):
    ideal = sorted(qrels.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(ideal))
    if idcg == 0:
        return 0.0
    return dcg_at_k(run, qrels, k) / idcg

In [44]:
import json
import numpy as np
from tqdm import tqdm
import time

corpus_map = {doc['id']: doc['text'] for doc in corpus}

def evaluate(retriever, queries, qrels, k=10, batch_size=64, save_path=None):
    ndcgs, recalls = [], []
    all_outputs = []

    q_texts = [q["text"] for q in queries]
    q_ids = [q["id"] for q in queries]
    all_runs = []

    start_time = time.time()
    if retriever.__class__.__name__ == 'BM25Retriever':
        for qtext in tqdm(q_texts, desc=f"BM25 Search"):
            all_runs.append(retriever.search(qtext, k=k))
    else:
        all_runs = retriever.search_batch(q_texts, k=k, batch_size=batch_size)

    end_time = time.time()
    total_time = end_time - start_time
    avg_latency_ms = (total_time / len(queries)) * 1000

    for i in tqdm(range(len(queries)), desc="Finalizing results"):
        qid = q_ids[i]
        qtext = q_texts[i]
        run = all_runs[i]
        qrel = qrels.get(qid, {})

        ndcgs.append(ndcg_at_k(run, qrel, k))
        recalls.append(recall_at_k(run, qrel, k))

        all_outputs.append({
            "query_id": qid,
            "query_text": qtext,
            "results": [{"doc_id": docid, "text": corpus_map.get(docid, "CONTENT NOT FOUND"), "score": score} for docid, score in run]
        })

    print("Total time: ", total_time)
    print("Average latency: ", avg_latency_ms)

    if save_path:
        print(f"Saving all {len(all_outputs)} query results to {save_path}...")
        with open(save_path, "w", encoding="utf-8") as f:
            for record in all_outputs:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Saved retrieval results successfully.")

    # Trả về kết quả trung bình
    return float(np.mean(ndcgs)), float(np.mean(recalls)), all_outputs

In [26]:
import gc

In [45]:
ndcg10, recall10, bm25_outputs = evaluate(bm25, queries, qrels, k=10, save_path=os.path.join(base_path, "data/ms_marco_val/bm25_results.jsonl"))
print(f"nDCG@10={ndcg10}, Recall@10={recall10}")

Finalizing results: 100%|██████████| 1000/1000 [00:00<00:00, 53831.79it/s]

Saving all 1000 query results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/bm25_results.jsonl...
Saved retrieval results successfully.
nDCG@10=0.06626940638895971, Recall@10=0.11333333333333333


In [11]:
del bm25
gc.collect()

In [31]:
embeddings_path = os.path.join(base_path, "data/ms_marco_val/corpus_embeddings_all-MiniLM-L6-v2.npy")
dense = DenseRetriever(corpus, embeddings_path=embeddings_path)

Loading pre-computed embeddings from /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/corpus_embeddings_all-MiniLM-L6-v2.npy...
Embeddings loaded successfully.


In [32]:
ndcg10, recall10, dense_outputs = evaluate(dense, queries, qrels, k=10, save_path=os.path.join(base_path, "data/ms_marco_val/dense_results.jsonl"))
print(f"nDCG@10={ndcg10}, Recall@10={recall10}")

Finalizing results: 100%|██████████| 1000/1000 [00:00<00:00, 62801.20it/s]

Saving all 1000 query results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/dense_results.jsonl...
Saved retrieval results successfully.
nDCG@10=0.24106656976026397, Recall@10=0.35566666666666663


In [33]:
del dense
gc.collect()

6118

In [34]:
def combine_and_evaluate_hybrid(bm25_outputs, dense_outputs, qrels, k=10, save_path=None, method='weighted_sum', alpha=0.5, rrf_k=60):
    """
    Hàm này không search lại. Nó chỉ lấy 2 bộ kết quả đã có,
    trộn chúng lại bằng phương pháp được chọn ('weighted_sum' hoặc 'rrf'),
    tính toán metrics và lưu file.
    """
    print(f"Combining BM25 and Dense results for Hybrid evaluation using '{method}' method...")
    ndcgs, recalls = [], []
    hybrid_outputs = []

    assert len(bm25_outputs) == len(dense_outputs)

    for i in tqdm(range(len(bm25_outputs)), desc=f"Combining Hybrid results ({method})"):
        bm_res = bm25_outputs[i]
        dn_res = dense_outputs[i]

        qid = bm_res["query_id"]
        qtext = bm_res["query_text"]
        qrel = qrels.get(qid, {})

        run = []

        if method == 'weighted_sum':
            bm_results_dict = {r['doc_id']: r['score'] for r in bm_res['results']}
            dn_results_dict = {r['doc_id']: r['score'] for r in dn_res['results']}

            bm_norm = normalize_scores(bm_results_dict)
            dn_norm = normalize_scores(dn_results_dict)
            all_ids = set(bm_norm.keys()) | set(dn_norm.keys())
            scores = {
                doc_id: alpha * dn_norm.get(doc_id, 0) + (1 - alpha) * bm_norm.get(doc_id, 0)
                for doc_id in all_ids
            }
            run = sorted(scores.items(), key=lambda item: item[1], reverse=True)[:k]

        elif method == 'rrf':
            bm_ranks = {r['doc_id']: i + 1 for i, r in enumerate(bm_res['results'])}
            dn_ranks = {r['doc_id']: i + 1 for i, r in enumerate(dn_res['results'])}

            all_ids = set(bm_ranks.keys()) | set(dn_ranks.keys())

            rrf_scores = {}
            for doc_id in all_ids:
                score = 0.0
                if doc_id in bm_ranks:
                    score += 1.0 / (rrf_k + bm_ranks[doc_id])
                if doc_id in dn_ranks:
                    score += 1.0 / (rrf_k + dn_ranks[doc_id])
                rrf_scores[doc_id] = score

            run = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:k]

        else:
            raise ValueError("Method must be 'weighted_sum' or 'rrf'")

        ndcgs.append(ndcg_at_k(run, qrel, k))
        recalls.append(recall_at_k(run, qrel, k))

        hybrid_outputs.append({
            "query_id": qid,
            "query_text": qtext,
            "results": [{"doc_id": docid, "text": corpus_map.get(docid, "CONTENT NOT FOUND"), "score": score} for docid, score in run]
        })

    if save_path:
        print(f"Saving all {len(hybrid_outputs)} hybrid query results to {save_path}...")
        with open(save_path, "w", encoding="utf-8") as f:
            for record in hybrid_outputs:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
        print(f"Saved hybrid results successfully.")

    return float(np.mean(ndcgs)), float(np.mean(recalls))

In [39]:
hybrid_ndcg, hybrid_recall = combine_and_evaluate_hybrid(
    bm25_outputs,
    dense_outputs,
    qrels,
    method='rrf',
    rrf_k=60,
    k=10,
    save_path=os.path.join(base_path, "data/ms_marco_val/hybrid_rrf_results.jsonl")
)
print(f"✅ Hybrid Results: nDCG@10 = {hybrid_ndcg:.4f}, Recall@10 = {hybrid_recall:.4f}")

Combining BM25 and Dense results for Hybrid evaluation using 'rrf' method...


Combining Hybrid results (rrf): 100%|██████████| 1000/1000 [00:00<00:00, 29800.52it/s]

Saving all 1000 hybrid query results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/hybrid_rrf_results.jsonl...
Saved hybrid results successfully.
✅ Hybrid Results: nDCG@10 = 0.1949, Recall@10 = 0.3207


In [40]:
hybrid_ndcg, hybrid_recall = combine_and_evaluate_hybrid(
    bm25_outputs,
    dense_outputs,
    qrels,
    alpha=0.7,
    method='weighted_sum',
    k=10,
    save_path=os.path.join(base_path, "data/ms_marco_val/hybrid_weighted_results.jsonl")
)
print(f"✅ Hybrid Results: nDCG@10 = {hybrid_ndcg:.4f}, Recall@10 = {hybrid_recall:.4f}")

Combining BM25 and Dense results for Hybrid evaluation using 'weighted_sum' method...


Combining Hybrid results (weighted_sum): 100%|██████████| 1000/1000 [00:00<00:00, 25816.03it/s]

Saving all 1000 hybrid query results to /content/drive/MyDrive/CapstoneProject/Retriever/data/ms_marco_val/hybrid_weighted_results.jsonl...
Saved hybrid results successfully.
✅ Hybrid Results: nDCG@10 = 0.2348, Recall@10 = 0.3427


# Đánh giá và Kết luận

## Bảng đánh giá
| Phương pháp | Thời gian Index/Embed | Thời gian Inference (1000 queries) | nDCG@10 | Recall@10 |
| :--- | :--- | :--- | :--- | :--- |
| **BM25** | Rất nhanh (vài phút, chỉ token hóa) | **44 phút 29 giây** - 1 CPU core | 0.0663 | 0.1133 |
| **Dense** | Rất lâu (20 phút cho lần đầu - giả định GPU T4) | **~6 giây** | **0.2411** | **0.3557** |
| **Hybrid (Weighted Sum)** | Không có (dùng lại) | ~1 giây (chỉ trộn kết quả) | 0.2348 | 0.3427 |
| **Hybrid (RRF)** | Không có (dùng lại) | ~1 giây (chỉ trộn kết quả) | 0.1949 | 0.3207 |

---

## **Giải thích các Metric**

### **1. Recall@10 (Độ phủ)**

*   Recall đo lường **khả năng tìm lại được tài liệu đúng**. Nó trả lời câu hỏi: "Trong tất cả các tài liệu đúng có thể có, model đã tìm thấy được bao nhiêu phần trăm trong top 10 kết quả?"
*   **Ví dụ dễ hiểu:** Giả sử cho một query, có tổng cộng 5 tài liệu được đánh dấu là "đúng".
    *   Nếu hệ thống trả về 10 kết quả, trong đó có 2 tài liệu đúng.
    *   Recall@10 sẽ là `2 / 5 = 0.4` (hay 40%).
*   `Recall@10 = 0.3557` của Dense là cao nhất, nghĩa là trung bình, nó tìm thấy được khoảng 35.6% số tài liệu đúng trong top 10 kết quả trả về.

### **2. nDCG@10 (Normalized Discounted Cumulative Gain)**

*   Đây là metric quan trọng nhất, nó đo lường **chất lượng xếp hạng**. Nó không chỉ quan tâm mô hình có tìm được tài liệu đúng hay không, mà còn quan tâm mô hình **xếp chúng ở vị trí nào**.
*   **Ví dụ dễ hiểu:** Google search. Một kết quả đúng ở vị trí #1 giá trị hơn rất nhiều so với một kết quả đúng ở vị trí #10. nDCG được thiết kế để phản ánh điều này.
    *   **Gain (Lợi ích):** Model được "điểm" khi tìm thấy một tài liệu đúng.
    *   **Cumulative (Tích lũy):** Điểm được cộng dồn cho tất cả các kết quả đúng trong top 10.
    *   **Discounted (Chiết khấu):** Điểm của một tài liệu đúng sẽ bị "giảm giá" (chiết khấu) nếu nó nằm ở vị trí thấp. Kết quả ở vị trí #1 được 100% điểm, vị trí #2 ít hơn, và cứ thế giảm dần.
    *   **Normalized (Chuẩn hóa):** Điểm số cuối cùng được chia cho điểm số của một bảng xếp hạng "hoàn hảo" lý tưởng, để đưa về thang điểm từ 0 đến 1. Điều này giúp so sánh công bằng giữa các query khác nhau.
*   `nDCG@10 = 0.2411` của Dense là cao nhất, cho thấy nó không chỉ tìm được nhiều tài liệu đúng (như Recall đã chỉ ra) mà còn xếp chúng ở những vị trí rất cao trong danh sách kết quả.

---

## **Giải thích các phương pháp Hybrid**

Cả hai phương pháp đều nhằm mục đích kết hợp kết quả từ BM25 (sparse) và Dense để tạo ra một danh sách xếp hạng cuối cùng tốt hơn.

### **1. Weighted Sum Fusion (Tổng có trọng số)**

*   **Cách hoạt động:** Nó hoạt động như một cái cân.
    1.  Nó lấy điểm số của BM25 và Dense.
    2.  Vì thang điểm của hai thằng này khác nhau (BM25 có thể > 20, Dense chỉ từ -1 đến 1), nó phải **chuẩn hóa (normalize)** cả hai về cùng một thang đo, thường là từ 0 đến 1.
    3.  Nó tính điểm cuối cùng bằng công thức: `Điểm cuối = (alpha * điểm_dense) + ((1 - alpha) * điểm_bm25)`.
*   **Ưu điểm:** Rất đơn giản, dễ hiểu. Tham số `alpha` giống như một cái núm vặn cho phép ta điều chỉnh "độ tin tưởng" vào mỗi retriever. Nếu mày thấy Dense tốt hơn, mày tăng `alpha` lên (ví dụ 0.8).
*   **Nhược điểm:** Rất nhạy cảm với cách chuẩn hóa điểm số. Nếu việc chuẩn hóa không tốt, một trong hai retriever có thể "lấn át" retriever còn lại một cách không công bằng. Kết quả (`0.2348`) khá tốt, cho thấy việc chuẩn hóa đã hoạt động tương đối ổn.

### **2. Reciprocal Rank Fusion (RRF - Hợp nhất Thứ hạng Nghịch đảo)**

*   **Cách hoạt động:** Phương pháp này **hoàn toàn không quan tâm đến điểm số**, nó chỉ quan tâm đến **thứ hạng (rank)**.
    1.  Nó không nhìn vào điểm `0.9` hay `25.0`. Nó chỉ nhìn: "Tài liệu A đứng thứ **1** trong danh sách của Dense và đứng thứ **5** trong danh sách của BM25".
    2.  Nó tính điểm cho mỗi tài liệu bằng công thức: `Điểm RRF = $$\frac(1 / (k + rank_dense)) + (1 / (k + rank_bm25))$$.
    3.  Hằng số `k` (thường là 60) được thêm vào để tránh việc các thứ hạng cao có điểm số quá chênh lệch so với các thứ hạng thấp.
*   **Ưu điểm:** Cực kỳ ổn định. Vì nó không dùng điểm số, nó miễn nhiễm với vấn đề thang đo khác nhau giữa BM25 và Dense. Đây thường là lựa chọn "an toàn" và hiệu quả khi ta không chắc chắn về chất lượng điểm số.
*   **Nhược điểm:** Nó bỏ qua thông tin về "mức độ tự tin". Ví dụ, Dense có thể rất chắc chắn về kết quả top 1 (điểm 0.99) so với top 2 (điểm 0.7), nhưng RRF chỉ coi chúng là hạng 1 và hạng 2. Trong trường hợp này, kết quả RRF (`0.1949`) thấp hơn Weighted Sum, có thể là do việc chuẩn hóa điểm số của Weighted Sum đã hoạt động tốt một cách bất ngờ, hoặc do RRF đã làm mất đi một số thông tin hữu ích từ điểm số của Dense.

## Giải thích kết quả

BM25 cho ra kết quả rất thấp. Kết quả của BM25 tệ đến mức nó trở thành "nhiễu" cho Hybrid.

BM25 thất bại trong trường hợp này vì 2 lý do chính sau:

### 1. Bộ dữ liệu được thiết kế để ưu ái Dense Retriever
Bộ dữ liệu MS MARCO này có vẻ được thiết kế để "ưu ái" (favor) Dense Retriever hơn là BM25.
Đây là những lý do chính:
- Bản chất của câu hỏi: MS MARCO được tạo ra với các câu hỏi thường là các câu hỏi tự nhiên, ví dụ: "What is the capital of France?". Các câu hỏi này đòi hỏi sự hiểu biết về ngữ nghĩa, từ đồng nghĩa, và mối quan hệ giữa các khái niệm. Với các câu hỏi tự nhiên trong MS MARCO, người dùng diễn đạt ý của họ theo nhiều cách.

  Dense Retriever, với khả năng "nhúng" (embed) các câu hỏi và tài liệu vào một không gian ngữ nghĩa, có thể nắm bắt được sự tương đồng về ý nghĩa, ngay cả khi không có từ khóa chung.

  Trong khi đó, BM25, với khả năng chỉ dựa vào từ khóa, sẽ gặp khó khăn trong việc hiểu được ý định thực sự của người hỏi.

- Đặc điểm của các tài liệu: Các tài liệu trong MS MARCO là các đoạn văn bản ngắn (passages). Các đoạn văn này thường chứa thông tin cô đọng, tập trung vào một chủ đề cụ thể. Điều này làm cho việc "nhúng" các đoạn văn thành các vector ý nghĩa trở nên dễ dàng hơn. BM25 có thể gặp khó khăn nếu các từ khóa trong câu hỏi không xuất hiện trực tiếp trong các đoạn văn, hoặc nếu các đoạn văn quá ngắn để chứa nhiều từ khóa.

## 2. Vấn đề từ vựng không khớp (Vocabulary Mismatch Problem)

Ngay cả khi không có từ đồng nghĩa, người ta vẫn có thể dùng các từ khác nhau để mô tả cùng một thứ.
- Query: "cách sửa lỗi rò rỉ ống nước"
- Tài liệu 1: "hướng dẫn khắc phục sự cố đường ống bị chảy nước"
- Tài liệu 2: "ống nước bị rò rỉ"

BM25 sẽ cho điểm Tài liệu 2 rất cao vì nó chứa chính xác các từ khóa ["ống", "nước", "rò", "rỉ"]. Nó sẽ cho điểm Tài liệu 1 rất thấp, vì các từ ["sửa", "lỗi"] không khớp với ["khắc", "phục", "sự", "cố"]. Dù tài liệu 1 mới là câu trả lời thực sự.

Dense Retriever: nó hiểu rằng "sửa lỗi", "khắc phục sự cố", "hướng dẫn" đều nằm trong một cụm ý nghĩa về "giải quyết vấn đề". Nó sẽ cho điểm cao cho cả hai, và có thể nhận ra Tài liệu 1 có giá trị hơn.


## Giải pháp
- Improve BM25 Retriever: ta đang đánh giá con cá khả năng leo cây. Vì vậy nên thay đổi bộ dataset để nó không bị bias towards any retrieval methods. Qua đây cũng cho thấy tầm quan trọng của việc xác định bài toán (Problem statements), data analysis, từ đó xác định usecase phù hợp và lựa chọn retriever phù hợp. Ngoài ra có thể sử dụng các thư viện hỗ trợ BM25 như Pyserini, các database tích hợp sẵn service search như ElasticSearch.
- Improve Dense Retriever: có thể sử dụng Milvus làm vector database hỗ trợ dense retriever query tốt hơn.
- Improve Hybrid Retriever:
  - Điều chỉnh trọng số alpha: Trong Weighted Sum, thử tăng alpha lên cao hơn (ví dụ 0.8 hoặc 0.9) để cho Dense có trọng số lớn hơn và giảm ảnh hưởng của BM25.
  - Chỉ dùng BM25 như một "bộ lọc": Một kỹ thuật khác là: lấy top 100 từ Dense, sau đó dùng BM25 để xếp hạng lại (re-rank) 100 kết quả đó. Cách này giới hạn phạm vi của BM25 và không để nó làm nhiễu kết quả.

## Kết luận
Đây không phải là một "lỗi" của BM25. Nó chỉ đơn giản là một sự khác biệt về cách tiếp cận. Trong một số trường hợp (ví dụ: tìm kiếm trong các tài liệu kỹ thuật, nơi từ khóa là quan trọng), BM25 vẫn có thể hoạt động tốt. Nhưng trong trường hợp của MS MARCO, Dense Retriever có lợi thế hơn hẳn. Đối với bộ dữ liệu MS MARCO, nơi các câu hỏi rất đa dạng và tự nhiên, sự "nông cạn" của BM25 bị bộc lộ hoàn toàn, dẫn đến kết quả rất tệ.